In [1]:
!pip install -q langchain langchain-community langchain-chroma
!pip install -q transformers sentence-transformers
!pip install -q faiss-cpu chromadb
!pip install -q accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

base_path = "/content/drive/MyDrive/mental-health-rag"

folders = ["data", "models", "notebooks", "outputs"]

for f in folders:
    os.makedirs(os.path.join(base_path, f), exist_ok=True)

print("✅ Folder structure created")

✅ Folder structure created


In [4]:
import langchain
import faiss
import chromadb
from sentence_transformers import SentenceTransformer

print("✅ All libraries working fine")

✅ All libraries working fine


In [5]:
!pip install chromadb --upgrade

**DATASET**

In [6]:
!pip install datasets

In [7]:
from datasets import load_dataset

dataset = load_dataset("Amod/mental_health_counseling_conversations")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Context', 'Response'],
        num_rows: 3512
    })
})


In [8]:
dataset['train'][0]

{'Context': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?",
 'Response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is someh

In [9]:
data = dataset['train']

documents = []

for item in data:
    text = f"User: {item['Context']}\nTherapist: {item['Response']}"
    documents.append(text)

print("Total documents:", len(documents))

Total documents: 3512


In [10]:
import os

save_path = "/content/drive/MyDrive/mental-health-rag/data/mental_health_docs.txt"

with open(save_path, "w") as f:
    for doc in documents:
        f.write(doc + "\n\n")

print("✅ Dataset saved successfully")

✅ Dataset saved successfully


**CHUNKING**

In [12]:
from langchain_community.document_loaders import TextLoader

file_path = "/content/drive/MyDrive/mental-health-rag/data/mental_health_docs.txt"

loader = TextLoader(file_path)
documents = loader.load()

print("Total documents:", len(documents))

Total documents: 1


In [13]:
print(documents[0].page_content[:500])

User: I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.
   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.
   How can I change my feeling of being worthless to everyone?
Therapist: If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a bi


In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Config A (small chunks)
splitter_500 = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Config B (large chunks)
splitter_1000 = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks_500 = splitter_500.split_documents(documents)
chunks_1000 = splitter_1000.split_documents(documents)

print("Chunks (500):", len(chunks_500))
print("Chunks (1000):", len(chunks_1000))

Chunks (500): 13394
Chunks (1000): 7622


In [16]:
print(chunks_500[0].page_content[:300])

User: I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.
   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.
   How can I change my feeli


**EMBEDDINGS**

In [17]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [18]:
# Model A (fast)
emb_model_1 = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Model B (better quality)
emb_model_2 = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

/tmp/ipykernel_25904/1145097654.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  emb_model_1 = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [19]:
text = "I feel anxious and stressed"

vector = emb_model_1.embed_query(text)

print("Vector length:", len(vector))
print(vector[:5])  # first 5 values

Vector length: 384
[0.06775378435850143, -0.0640564039349556, -0.004080289509147406, 0.046576518565416336, 0.056049685925245285]


In [20]:
# For chunk set A
texts_500 = [doc.page_content for doc in chunks_500]
embeddings_500 = emb_model_1.embed_documents(texts_500)

print("Embeddings created:", len(embeddings_500))

Embeddings created: 13394


In [23]:
texts_1000 = [doc.page_content for doc in chunks_1000]
embeddings_1000 = emb_model_2.embed_documents(texts_1000)

print("Embeddings created:", len(embeddings_1000))

Embeddings created: 7622


**VECTOR DATABASE**

In [24]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores import Chroma

In [25]:
# Using small chunks + embedding model 1
db_faiss = FAISS.from_documents(
    chunks_500,
    emb_model_1
)

print("✅ FAISS DB created")

✅ FAISS DB created


In [26]:
db_chroma = Chroma.from_documents(
    chunks_500,
    emb_model_1,
    persist_directory="/content/drive/MyDrive/mental-health-rag/chroma_db"
)

print("✅ Chroma DB created")

✅ Chroma DB created


In [27]:
query = "I feel stressed and anxious"

docs_faiss = db_faiss.similarity_search(query, k=3)

for i, doc in enumerate(docs_faiss):
    print(f"\nResult {i+1}:\n", doc.page_content[:200])


Result 1:
 What can I do to manage my stress?

Result 2:
 What can I do to manage my stress?

Result 3:
 What can I do to manage my stress?


In [28]:
docs_chroma = db_chroma.similarity_search(query, k=3)

for i, doc in enumerate(docs_chroma):
    print(f"\nResult {i+1}:\n", doc.page_content[:200])


Result 1:
 What can I do to manage my stress?

Result 2:
 What can I do to manage my stress?

Result 3:
 What can I do to manage my stress?


In [29]:
print("FAISS Results:", len(docs_faiss))
print("Chroma Results:", len(docs_chroma))

FAISS Results: 3
Chroma Results: 3


In [41]:
query = "I feel very anxious and can't sleep"

print("FAISS Response:\n")
print(rag_chat(query, retriever_faiss))

print("\n\nChroma Response:\n")
print(rag_chat(query, retriever_chroma))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FAISS Response:



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a mental health support assistant.

Rules:
- Be empathetic and supportive
- Do NOT give medical or harmful advice
- If unsure, say "I am not fully certain"
- Encourage seeking professional help when needed

Context:
User: I've been having horrible anxiety for the last week. I can't sleep. I get a sense of doom, and it's hard to breathe. I feel like nothing I do makes it better.
Therapist: It sounds like you are noticing yourself becoming overwhelmed with anxiety, feeling more irritable, and struggling to sleep consistently. There are many possibilities, in regards to what may be contributing to these things you are noticing, and a competent therapist may be able to help. In therapy, you may be able to gain insight into these experiences as well as develop strategies for coping with and eventually alleviating anxiety, irritability, and inconsistent sleep.
Therapist: It sounds like you are noticing yourself becoming overwhelmed with anxiety, feeling more irritable, and strugglin

**PROMPT + LLM (CORE OF RAG)**

In [30]:
retriever_faiss = db_faiss.as_retriever(search_kwargs={"k": 3})
retriever_chroma = db_chroma.as_retriever(search_kwargs={"k": 3})

In [53]:
def create_prompt(context, query):
    return f"""
You are a compassionate mental health assistant.

Guidelines:
- Respond in a calm, supportive, and human-like way
- Do NOT repeat the question
- Do NOT repeat sentences
- Keep answer concise (3–5 sentences)
- If user expresses distress, respond with empathy

Context:
{context}

User: {query}

Helpful Response:
"""

In [33]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=200
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [34]:
print(llm("I feel anxious and stressed")[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I feel anxious and stressed.”


If you have any questions or concerns about this article, please don't hesitate to contact me via Twitter (@BetsyH_) or Facebook (facebook.com/BetsyH_)


In [35]:
model="tiiuae/falcon-rw-1b"

In [40]:
def rag_chat(query, retriever):
    docs = retriever.invoke(query)

    context = "\n".join([doc.page_content for doc in docs])

    prompt = create_prompt(context, query)

    response = llm(prompt)[0]['generated_text']

    return response

In [38]:
def rag_chat(query, retriever):

    docs = retriever.invoke(query)   # ✅ FIXED

    context = "\n".join([doc.page_content for doc in docs])

    prompt = create_prompt(context, query)

    response = llm(prompt)[0]['generated_text']

    return response

In [43]:
def safety_check(query):
    danger_words = ["suicide", "kill myself", "self-harm"]

    for word in danger_words:
        if word in query.lower():
            return True
    return False

In [54]:
def rag_chat_safe(query, retriever):

    if safety_check(query):
        return """I'm really sorry you're feeling this way.
You are not alone. Please reach out to someone you trust.

📞 India Helpline: 9152987821 (AASRA)
"""

    docs = retriever.invoke(query)

    context = "\n".join([doc.page_content for doc in docs])

    prompt = create_prompt(context, query)

    response = llm(prompt)[0]['generated_text']

    # Remove prompt from output
    response = response.replace(prompt, "").strip()

    return response

**EVALUATION + UI + FINALIZATION**

In [45]:
!pip install nltk rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=37852080040dfcaafa8b25ccf3df246636c2ee703abd9bb16e6937c542c5d544
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [46]:
from nltk.translate.bleu_score import sentence_bleu

reference = ["I'm sorry you're feeling anxious. Try some relaxation techniques."]
candidate = rag_chat("I feel anxious", retriever_faiss)

score = sentence_bleu([reference[0].split()], candidate.split())

print("BLEU Score:", score)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BLEU Score: 1.0620962713238408e-155


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [47]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

scores = scorer.score(reference[0], candidate)

print(scores)

{'rouge1': Score(precision=0.01507537688442211, recall=0.5454545454545454, fmeasure=0.029339853300733496), 'rougeL': Score(precision=0.01507537688442211, recall=0.5454545454545454, fmeasure=0.029339853300733496)}


In [49]:
def rag_chat_safe(query, retriever):

    # Safety check
    if safety_check(query):
        return "I'm really sorry you're feeling this way. Please consider reaching out to a trusted person or a helpline."

    # ✅ FIXED retrieval
    docs = retriever.invoke(query)

    context = "\n".join([doc.page_content for doc in docs])

    prompt = create_prompt(context, query)

    response = llm(prompt)[0]['generated_text']

    # Optional cleanup
    response = response[len(prompt):]

    return response

In [55]:
queries = [
    "I feel depressed",
    "I have anxiety",
    "I want to harm myself"
]

for q in queries:
    print("\nQuery:", q)
    print(rag_chat_safe(q, retriever_faiss))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Query: I feel depressed


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hi there! You sound as if depression has been present for some time now; it's not surprising to hear about how hard things can be when we're feeling low or down - especially at this stage where our thoughts might also focus on trying harder instead finding solutions from within ourselves..The world famous “Swan Song Festival – A Tribute To The Beatles' Music" will take place June 11th through 13rd 2017at New York City Center Studios which includes performances featuring Broadway stars alongside an amazing cast including Tony winner Kelli O”Hara who plays George Harrison along side Patti LuPone whose rendition was nominated Best Performance Of Her Career 2016 Grammy Awards. We had previously seen her sing live during last year tribute concert held September

Query: I have anxiety


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hello there! Thank for reaching us at this time!! We would like so much if we could help by providing some information about Anxiety? Have an article that helps explain what is going on here… Checkout our blog post below.. Feel free …“The greatest gift God has given mankind was His Son Jesus Christ who died as payment for sin; He rose again from death after 3 days but many Christians don't know how they can receive salvation through faith because their minds haven't been enlightened yet." This statement will make people think twice before speaking ill towards others even though most do not understand Christian theology very well themselves...but then why should one criticize another person just based off ignorance?! To learn more please read ‘What Is The Bible And

Query: I want to harm myself
I am so sorry that you’re struggling right now! It sounds like your thoughts of harming yourself can be triggered by many different things – such as sadness/depression from others around us? Or i

In [51]:
model="tiiuae/falcon-rw-1b"

In [52]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="tiiuae/falcon-rw-1b",
    max_new_tokens=150,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.2
)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.62G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample', 'temperature', 'repetition_penalty', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [56]:
llm = pipeline(
    "text-generation",
    model="tiiuae/falcon-rw-1b",
    max_new_tokens=150,
    pad_token_id=50256
)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [57]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 126.5 MB/s eta 0:00:00


In [58]:
%%writefile app.py
import streamlit as st

st.title("🧠 Mental Health Chatbot")

query = st.text_input("How are you feeling today?")

if query:
    response = rag_chat_safe(query, retriever_faiss)
    st.write(response)

Writing app.py
